# Capability Design

Capability는 단순한 도구 이름이 아니라 `리소스`, `행위`, `목적`, `민감도`, `사람 검토 조건`을 포함한 권한 단위입니다.


In [ ]:
from dataclasses import dataclass
from pprint import pprint
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


## 1단계: Capability 카탈로그

Capability ID는 `도메인.리소스.행위` 형태로 작성하면 운영 중 검색과 감사를 하기 쉽습니다.


In [ ]:
Sensitivity = Literal["public", "internal", "restricted"]


@dataclass(frozen=True)
class Capability:
    id: str
    owner_agent: str
    resource: str
    action: str
    sensitivity: Sensitivity
    allowed_purposes: frozenset[str]
    requires_human: bool = False


CAPABILITIES = {
    "public.docs.search": Capability(
        id="public.docs.search",
        owner_agent="research_agent",
        resource="public_docs",
        action="search",
        sensitivity="public",
        allowed_purposes=frozenset({"research", "support"}),
    ),
    "finance.invoice.read": Capability(
        id="finance.invoice.read",
        owner_agent="finance_agent",
        resource="invoice",
        action="read",
        sensitivity="internal",
        allowed_purposes=frozenset({"billing_support"}),
    ),
    "finance.refund.submit": Capability(
        id="finance.refund.submit",
        owner_agent="finance_agent",
        resource="refund",
        action="submit",
        sensitivity="restricted",
        allowed_purposes=frozenset({"billing_support"}),
        requires_human=True,
    ),
    "security.incident.create": Capability(
        id="security.incident.create",
        owner_agent="security_agent",
        resource="incident",
        action="create",
        sensitivity="internal",
        allowed_purposes=frozenset({"security_response"}),
    ),
}


ROLE_CLEARANCE = {
    "guest": {"public"},
    "employee": {"public", "internal"},
    "security_admin": {"public", "internal", "restricted"},
}


## 2단계: 정책 상태와 판단 함수

정책 엔진은 `approved`, `denied`, `needs_review` 중 하나를 반환합니다.


In [ ]:
class CapabilityState(TypedDict, total=False):
    agent_id: str
    user_role: Literal["guest", "employee", "security_admin"]
    requested_capability: str
    purpose: str
    payload: dict
    decision: Literal["approved", "denied", "needs_review"]
    reason: str
    execution_result: str
    review_ticket: str
    audit_log: list[str]


def policy_node(state: CapabilityState) -> CapabilityState:
    capability_id = state["requested_capability"]
    capability = CAPABILITIES.get(capability_id)
    audit = state.get("audit_log", [])

    if capability is None:
        return {
            **state,
            "decision": "denied",
            "reason": "unknown_capability",
            "audit_log": audit + [f"policy:denied:{capability_id}:unknown"],
        }

    if capability.owner_agent != state["agent_id"]:
        return {
            **state,
            "decision": "denied",
            "reason": "wrong_agent_owner",
            "audit_log": audit + [f"policy:denied:{capability_id}:owner_mismatch"],
        }

    if state["purpose"] not in capability.allowed_purposes:
        return {
            **state,
            "decision": "denied",
            "reason": "purpose_not_allowed",
            "audit_log": audit + [f"policy:denied:{capability_id}:purpose"],
        }

    clearance = ROLE_CLEARANCE[state["user_role"]]
    if capability.sensitivity not in clearance:
        return {
            **state,
            "decision": "denied",
            "reason": "insufficient_clearance",
            "audit_log": audit + [f"policy:denied:{capability_id}:clearance"],
        }

    if capability.requires_human:
        return {
            **state,
            "decision": "needs_review",
            "reason": "human_approval_required",
            "audit_log": audit + [f"policy:review:{capability_id}"],
        }

    return {
        **state,
        "decision": "approved",
        "reason": "policy_passed",
        "audit_log": audit + [f"policy:approved:{capability_id}"],
    }


def route_by_policy(state: CapabilityState) -> str:
    return state["decision"]


## 3단계: 승인, 거절, 사람 검토 노드


In [ ]:
def execute_capability(state: CapabilityState) -> CapabilityState:
    capability_id = state["requested_capability"]
    return {
        **state,
        "execution_result": f"{capability_id} 실행 완료",
        "audit_log": state.get("audit_log", []) + [f"execute:{capability_id}"],
    }


def create_review_ticket(state: CapabilityState) -> CapabilityState:
    capability_id = state["requested_capability"]
    ticket_id = f"REVIEW-{abs(hash((capability_id, state['purpose']))) % 10000:04d}"
    return {
        **state,
        "review_ticket": ticket_id,
        "execution_result": "자동 실행 보류",
        "audit_log": state.get("audit_log", []) + [f"review_ticket:{ticket_id}"],
    }


def deny_capability(state: CapabilityState) -> CapabilityState:
    return {
        **state,
        "execution_result": f"거절: {state['reason']}",
        "audit_log": state.get("audit_log", []) + [f"deny:{state['reason']}"],
    }


In [ ]:
workflow = StateGraph(CapabilityState)

workflow.add_node("policy_node", policy_node)
workflow.add_node("execute_capability", execute_capability)
workflow.add_node("create_review_ticket", create_review_ticket)
workflow.add_node("deny_capability", deny_capability)

workflow.add_edge(START, "policy_node")
workflow.add_conditional_edges(
    "policy_node",
    route_by_policy,
    {
        "approved": "execute_capability",
        "needs_review": "create_review_ticket",
        "denied": "deny_capability",
    },
)
workflow.add_edge("execute_capability", END)
workflow.add_edge("create_review_ticket", END)
workflow.add_edge("deny_capability", END)

graph = workflow.compile()


## 4단계: 정책 테스트

같은 도구라도 Agent, 목적, 사용자 역할, 사람 검토 조건에 따라 결과가 달라집니다.


In [ ]:
examples = [
    {
        "agent_id": "research_agent",
        "user_role": "guest",
        "requested_capability": "public.docs.search",
        "purpose": "research",
        "payload": {"query": "LangGraph"},
    },
    {
        "agent_id": "research_agent",
        "user_role": "employee",
        "requested_capability": "finance.invoice.read",
        "purpose": "billing_support",
        "payload": {"account_id": "C-100"},
    },
    {
        "agent_id": "finance_agent",
        "user_role": "employee",
        "requested_capability": "finance.refund.submit",
        "purpose": "billing_support",
        "payload": {"account_id": "C-100", "amount": 50000},
    },
    {
        "agent_id": "finance_agent",
        "user_role": "guest",
        "requested_capability": "finance.invoice.read",
        "purpose": "billing_support",
        "payload": {"account_id": "C-100"},
    },
]

for example in examples:
    result = graph.invoke({**example, "audit_log": []})
    print("=" * 80)
    print("Capability:", result["requested_capability"])
    print("Decision:", result["decision"])
    print("Reason:", result["reason"])
    print("Result:", result["execution_result"])
    pprint(result["audit_log"])


## 정리

- Capability는 실행 권한의 최소 단위입니다.
- 목적, 민감도, 사람 검토 조건을 Capability에 포함시키면 정책 판단을 재사용할 수 있습니다.
- `needs_review`는 실패가 아니라 안전한 운영 경로입니다.
